# Download All UBL 2.4 Google Sheets Revisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/setup-ubl-sheets-access-zWGFj/notebooks/download-ubl24-revisions.ipynb)

Downloads every revision of the UBL 2.4 Google Sheets as ODS,
gzips, and saves to Drive.

**Crash-resilient:** re-run and it picks up where it left off.

## How it works

Uses the direct Sheets export URL to access **all** internal revisions
(not just the ~12-16 returned by `Drive API revisions.list`):

```
https://docs.google.com/spreadsheets/export?id=ID&revision=N&exportFormat=ods
```

Iterates revision numbers 1 → max (oldest → newest). For each:
1. Export as ODS via direct URL
2. Hash `content.xml` to detect unique spreadsheet states
3. Save `rev-{N}.ods.gz` to Drive
4. Record in `checkpoint-{sheet}.json`

Non-existent revision numbers return 400/404 — simply skipped.

## Output

```
Drive: ubl-gc-revisions/
├── ubl24_library/
│   ├── rev-1.ods.gz
│   ├── rev-2.ods.gz
│   └── ...
├── ubl24_documents/
│   └── ...
├── checkpoint-ubl24_library.json
└── checkpoint-ubl24_documents.json
```

## Differences from UBL 2.5

- **No Endorsed entities** — Endorsed is new in 2.5
- **23 columns** in Entities (vs 27 in 2.5)
- **Signature entities** exist (same structure as 2.5)
- **Download only** — GC conversion handled separately

In [ ]:
# === Step 0: Auth ===
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest
from datetime import datetime, timezone, UTC

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')
if creds.expiry:
    remaining = (creds.expiry - datetime.now(UTC).replace(tzinfo=None)).total_seconds()
    print(f'Token expiry: {creds.expiry.isoformat()} ({remaining:.0f}s from now)')

In [ ]:
# === Step 1: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

In [ ]:
# === Step 2: Configuration & helpers ===
import json, hashlib, gzip, time, zipfile, io, re
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

SHEETS = {
    'ubl24_library':   {'id': '1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs'},
    'ubl24_documents': {'id': '1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y'},
}

# Refresh token when within this many seconds of expiry.
# Google OAuth tokens last 3600s; refreshing at 300s margin means
# the token is guaranteed valid for at least 5 more minutes.
TOKEN_REFRESH_MARGIN = 300  # 5 minutes


def _utcnow_naive():
    """Naive UTC now — matches creds.expiry which is naive UTC."""
    return datetime.now(UTC).replace(tzinfo=None)


def ensure_fresh_token():
    """Refresh TOKEN if expired or within TOKEN_REFRESH_MARGIN of expiry.

    Key fix: sets creds.token = None BEFORE calling creds.refresh() to
    force Google to mint a brand-new access token. Without this, refresh()
    may silently return the same still-valid token.
    """
    global TOKEN

    if not creds.expiry:
        return

    remaining = (creds.expiry - _utcnow_naive()).total_seconds()

    if remaining > TOKEN_REFRESH_MARGIN:
        return

    old_token_prefix = TOKEN[:8] if TOKEN else '(none)'
    reason = 'expired' if remaining <= 0 else f'{remaining:.0f}s left'

    creds.token = None  # force new token from endpoint
    creds.refresh(AuthRequest())
    TOKEN = creds.token

    new_token_prefix = TOKEN[:8] if TOKEN else '(none)'
    new_remaining = (creds.expiry - _utcnow_naive()).total_seconds()
    changed = 'NEW' if new_token_prefix != old_token_prefix else 'SAME!'

    print(f'\n  >> TOKEN REFRESH ({reason}): '
          f'{old_token_prefix}->>{new_token_prefix} [{changed}] '
          f'valid {new_remaining:.0f}s')


def authenticated_get(url, binary=True):
    """Authenticated GET with token refresh + retry + exponential backoff.
    Returns (status_code, data_bytes) or (status_code, None)."""
    global TOKEN
    ensure_fresh_token()
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                return resp.status, resp.read()
        except HTTPError as e:
            if e.code == 401:
                old_pfx = TOKEN[:8] if TOKEN else '(none)'
                creds.token = None
                creds.refresh(AuthRequest())
                TOKEN = creds.token
                new_pfx = TOKEN[:8] if TOKEN else '(none)'
                print(f'  [401 refresh: {old_pfx}->>{new_pfx}]', end=' ')
                headers = {'Authorization': f'Bearer {TOKEN}'}
                continue
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code})...', end='')
                time.sleep(wait)
                continue
            return e.code, None
        except Exception as e:
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue
            return 0, None
    return 0, None


def export_revision_ods(sheet_id, rev_num):
    """Export a specific revision as ODS using the direct Sheets URL.
    Returns ODS bytes or None (if revision doesn't exist)."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={sheet_id}&revision={rev_num}&exportFormat=ods')
    status, data = authenticated_get(url)
    if status == 200 and data and len(data) > 500:
        return data
    return None


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


print('All helpers ready')

## Step 3: Discover max revision for each sheet

Uses **Drive API v2 `revisions.list`** to get the highest revision ID
directly — no binary search needed. The v2 revision IDs are the same
sequential integers used by the export URL.

Falls back to binary search if the API call fails.

In [ ]:
def get_max_revision_via_api(sheet_id):
    """Get max revision using Drive API v2 revisions.list.
    
    The v2 API returns revisions with integer IDs that match
    the sequential revision numbers used by the export URL.
    Returns (max_rev, revision_list) or (None, []) on failure.
    """
    url = (f'https://www.googleapis.com/drive/v2/files/{sheet_id}'
           f'/revisions?fields=items(id,modifiedDate)')
    status, data = authenticated_get(url)
    if status != 200 or not data:
        return None, []

    result = json.loads(data.decode('utf-8'))
    items = result.get('items', [])
    if not items:
        return None, []

    # IDs are integer strings — find the max
    max_id = max(int(r['id']) for r in items)
    print(f'  API returned {len(items)} revisions, highest ID: {max_id}')
    for r in items[-5:]:
        print(f'    rev-{r["id"]}  {r.get("modifiedDate", "?")[:19]}')
    return max_id, items


def find_max_revision_binary(sheet_id, start_guess=100):
    """Binary search fallback for the highest valid revision number."""
    lo = 1
    hi = start_guess
    print(f'  Binary search: finding upper bound (starting at {hi})...', end='')

    while True:
        url = (f'https://docs.google.com/spreadsheets/export'
               f'?id={sheet_id}&revision={hi}&exportFormat=ods')
        status, data = authenticated_get(url)
        if status == 200 and data and len(data) > 500:
            print(f' {hi}=OK', end='')
            lo = hi
            hi *= 2
        else:
            print(f' {hi}=FAIL')
            break
        if hi > 100000:
            print(f'  WARNING: exceeded 100k, stopping')
            return hi

    print(f'  Bisecting [{lo}..{hi}]...', end='')
    while lo < hi - 1:
        mid = (lo + hi) // 2
        url = (f'https://docs.google.com/spreadsheets/export'
               f'?id={sheet_id}&revision={mid}&exportFormat=ods')
        status, data = authenticated_get(url)
        if status == 200 and data and len(data) > 500:
            lo = mid
            print(f' {mid}=OK', end='')
        else:
            hi = mid
            print(f' {mid}=FAIL', end='')

    print(f'\n  Max revision (binary): {lo}')
    return lo


# --- Discover max_rev for each sheet ---
for sheet_key, info in SHEETS.items():
    print(f'\n{sheet_key} ({info["id"][:12]}...):')

    api_max, api_revs = get_max_revision_via_api(info['id'])

    if api_max:
        # Verify the API result with one export probe
        print(f'  Verifying rev-{api_max} via export...', end=' ')
        test = export_revision_ods(info['id'], api_max)
        if test:
            print(f'{len(test):,} bytes OK')
            info['max_rev'] = api_max
        else:
            print(f'FAIL — falling back to binary search')
            info['max_rev'] = find_max_revision_binary(info['id'],
                                                        start_guess=api_max)
    else:
        print(f'  API failed — using binary search')
        info['max_rev'] = find_max_revision_binary(info['id'])

    info['api_revisions'] = api_revs

print(f'\n{"="*60}')
print(f'Max revisions discovered:')
for sheet_key, info in SHEETS.items():
    print(f'  {sheet_key}: {info["max_rev"]}')

In [ ]:
# === Step 4: Quick probe — verify export URL works ===
# Test with the discovered max revision of each sheet
for sheet_key, info in SHEETS.items():
    print(f'{sheet_key}: testing rev-{info["max_rev"]}...', end=' ')
    test = export_revision_ods(info['id'], info['max_rev'])
    if test:
        print(f'{len(test):,} bytes OK')
    else:
        print('FAILED — check auth or sheet ID')

## Step 5: Download all revisions

Processes **both sheets** automatically (library first, then documents).
No configuration needed — just Run All.

Each revision is exported as ODS, gzipped, and saved to Drive.
Content hashes track unique spreadsheet states.

In [ ]:
for SHEET_KEY, info in SHEETS.items():
    sheet_id = info['id']
    max_rev = info['max_rev']

    # --- Directories ---
    ods_dir = DRIVE_DIR / SHEET_KEY
    ods_dir.mkdir(exist_ok=True)

    # --- Resume from checkpoint ---
    checkpoint_path = DRIVE_DIR / f'checkpoint-{SHEET_KEY}.json'
    if checkpoint_path.exists():
        checkpoint = json.loads(checkpoint_path.read_text())
        print(f'Resuming {SHEET_KEY} from checkpoint: '
              f'last_rev={checkpoint.get("last_rev", "?")}')
    else:
        checkpoint = {
            'sheet': SHEET_KEY,
            'sheet_id': sheet_id,
            'max_rev': max_rev,
            'revisions': [],
        }

    done_revs = {r['rev'] for r in checkpoint.get('revisions', [])}
    seen_hashes = set()
    hash_to_rev = {}
    for r in checkpoint.get('revisions', []):
        h = r.get('content_hash')
        if h:
            seen_hashes.add(h)
            if h not in hash_to_rev:
                hash_to_rev[h] = r['rev']

    # --- Main loop ---
    downloaded = 0
    skipped_done = 0
    skipped_missing = 0
    consecutive_missing = 0

    # Proven safe rate from method-c validation: 10s between requests.
    # Lower values cause frequent 429s from the Google Sheets export API.
    REQUEST_DELAY = 10.0

    print(f'\n{"="*60}')
    print(f'{SHEET_KEY}: processing rev 1..{max_rev} (delay={REQUEST_DELAY}s)')
    print(f'{"="*60}\n')

    for rev_num in range(1, max_rev + 1):
        if rev_num in done_revs:
            skipped_done += 1
            continue

        gz_path = ods_dir / f'rev-{rev_num}.ods.gz'
        if gz_path.exists() and gz_path.stat().st_size > 0:
            skipped_done += 1
            done_revs.add(rev_num)
            continue

        ods_data = export_revision_ods(sheet_id, rev_num)
        if not ods_data:
            skipped_missing += 1
            consecutive_missing += 1
            if consecutive_missing % 100 == 0:
                print(f'  [{rev_num}/{max_rev}] '
                      f'{skipped_missing} missing so far...')
                time.sleep(1)
            continue

        consecutive_missing = 0
        pct = rev_num / max_rev * 100
        print(f'[{rev_num}/{max_rev} {pct:.0f}%]', end=' ')

        content_hash = ods_content_hash(ods_data)
        is_new = content_hash and content_hash not in seen_hashes
        if content_hash:
            seen_hashes.add(content_hash)

        gz_data = gzip.compress(ods_data, compresslevel=6)
        gz_path.write_bytes(gz_data)

        entry = {
            'rev': rev_num,
            'ods_size': len(ods_data),
            'gz_size': len(gz_data),
            'content_hash': content_hash,
            'is_new_state': is_new,
        }

        if not is_new:
            ref = hash_to_rev.get(content_hash)
            if ref:
                entry['same_as'] = ref
            print(f'{len(ods_data):,}b (same as rev-{ref})' if ref
                  else f'{len(ods_data):,}b')
        else:
            hash_to_rev[content_hash] = rev_num
            print(f'{len(ods_data):,}b NEW unique state #{len(seen_hashes)}')

        checkpoint['revisions'].append(entry)
        done_revs.add(rev_num)
        downloaded += 1

        if downloaded % 25 == 0:
            checkpoint['last_rev'] = rev_num
            checkpoint['delay'] = REQUEST_DELAY
            checkpoint['timestamp'] = time.strftime('%Y-%m-%dT%H:%M:%SZ',
                                                    time.gmtime())
            checkpoint_path.write_text(json.dumps(checkpoint, indent=2))
            print(f'  --- checkpoint: {downloaded} dl, '
                  f'{skipped_missing} miss, '
                  f'{len(seen_hashes)} unique ---')

        time.sleep(REQUEST_DELAY)

    # --- Final save for this sheet ---
    checkpoint['last_rev'] = max_rev
    checkpoint['unique_states'] = len(seen_hashes)
    checkpoint['max_rev'] = max_rev
    checkpoint['total_downloaded'] = downloaded + skipped_done
    checkpoint['total_missing'] = skipped_missing
    checkpoint['timestamp'] = time.strftime('%Y-%m-%dT%H:%M:%SZ',
                                            time.gmtime())
    checkpoint_path.write_text(json.dumps(checkpoint, indent=2))

    print(f'\n{"="*60}')
    print(f'DONE: {SHEET_KEY} (rev 1..{max_rev})')
    print(f'  Downloaded:       {downloaded}')
    print(f'  Already done:     {skipped_done}')
    print(f'  Missing/skipped:  {skipped_missing}')
    print(f'  Unique states:    {len(seen_hashes)}')
    print(f'  Checkpoint:       {checkpoint_path}')
    print()

print('ALL SHEETS COMPLETE')

## Step 6: Analyze Results

In [ ]:
for sheet_key in SHEETS:
    cp = DRIVE_DIR / f'checkpoint-{sheet_key}.json'
    if not cp.exists():
        print(f'{sheet_key}: not yet downloaded')
        continue

    c = json.loads(cp.read_text())
    revs = c.get('revisions', [])
    hash_counts = Counter(
        r['content_hash'] for r in revs if r.get('content_hash')
    )

    print(f'\n{"="*60}')
    print(f'{sheet_key}: {len(revs)} downloaded, '
          f'{len(hash_counts)} unique states, '
          f'{c.get("total_missing", "?")} missing')
    print(f'{"="*60}')

    print(f'\nUnique states (most common first):')
    for rank, (h, count) in enumerate(hash_counts.most_common(20), 1):
        matching = sorted(
            [r for r in revs if r.get('content_hash') == h],
            key=lambda r: r['rev']
        )
        first = matching[0]
        last = matching[-1]
        print(f'  {rank:3d}. {h[:16]}... x{count:4d}  '
              f'rev-{first["rev"]} to rev-{last["rev"]}')

    if len(hash_counts) > 20:
        print(f'  ... and {len(hash_counts) - 20} more unique states')

    # Timeline of unique state changes
    new_states = sorted(
        [r for r in revs if r.get('is_new_state')],
        key=lambda r: r['rev']
    )
    if new_states:
        print(f'\nUnique state transitions ({len(new_states)} changes):')
        for r in new_states:
            print(f'  rev-{r["rev"]:5d}  {r["ods_size"]:>9,}b  '
                  f'{r["content_hash"][:16]}...')

## What Next

After both sheets are downloaded:

1. **Inspect unique states** — how many distinct editing states exist?
2. **Compare with known releases** — match against the 4 official
   UBL 2.4 releases (CSD01, CSD02, CS01, OS)
3. **GC conversion** can be done separately using Saxon/Crane with
   UBL 2.4 parameters (23-column Entities, no Endorsed file)
4. **Cross-reference with 2.5** — the 2.4 CSD02 master sheet edits
   may show the transition from 2.3→2.4 and 2.4→2.5

---

## Monitoring from outside Colab

The notebook writes everything to Google Drive at:
```
My Drive/ubl-gc-revisions/
```

This folder is shared publicly (read-only), so you can check progress
without Colab access.

### Drive folder structure

```
ubl-gc-revisions/                          ← root folder (public)
├── checkpoint-ubl24_library.json          ← current position & rate state
├── checkpoint-ubl24_documents.json
├── ubl24_library/                         ← ODS files: rev-1.ods.gz .. rev-N.ods.gz
└── ubl24_documents/                       ← ODS files: rev-1.ods.gz .. rev-N.ods.gz
```

Note: The JSON files are created on first run. If the Drive is clean
(no JSON files yet), that's normal — run the notebook once.

### What each file tells you

| File | Key fields | What it means |
|------|-----------|---------------|
| **checkpoint-ubl24_library.json** | `last_rev` | Last revision processed. Compare to `max_rev` for progress. |
| | `max_rev` | Total revisions to process (discovered at runtime via API) |
| | `unique_states` | How many distinct spreadsheet states found so far |
| | `total_downloaded` | Revisions successfully downloaded |
| | `total_missing` | Revision numbers that returned 400/404 (gaps) |
| | `timestamp` | When the checkpoint was last written |
| **checkpoint-ubl24_documents.json** | *(same fields)* | Same structure for the documents sheet |

The checkpoint also contains a `revisions` array with per-revision detail:
- `rev`: revision number
- `ods_size` / `gz_size`: raw and compressed sizes
- `content_hash`: SHA256 of content.xml (for dedup)
- `is_new_state`: true if this was a previously-unseen hash

### Quick progress check

The checkpoint is the fastest way to see where the run is:

```bash
# 1. List files in the root folder to find file IDs
python3 work-sheets/scripts/explore-public-drive.py \
    1SVXV_8CF4ib9YsVZ6G7AqIP3gNK6q3Gj /tmp/drive-listing.json

# 2. Download checkpoint by its file ID (from the listing)
curl -sL "https://drive.google.com/uc?export=download&id=FILE_ID" \
    | python3 -m json.tool

# 3. Quick math: progress = last_rev / max_rev
#    (max_rev is discovered at runtime — check the checkpoint JSON)
```

### Folder IDs (for direct access)

| What | ID |
|------|-----|
| **Root folder** | `1SVXV_8CF4ib9YsVZ6G7AqIP3gNK6q3Gj` |

Note: The `ubl24_library/` and `ubl24_documents/` subfolders are created
on first run. Use the explore script to discover their IDs after the
notebook has started.

### Done when

Both checkpoints show `last_rev` equal to `max_rev`. The notebook
processes library first, then documents, and prints
`ALL SHEETS COMPLETE` when both are done.

### Rate limiting

Uses a **10-second delay** between requests — the proven safe rate from
the method-c validation notebook. The Google Sheets export API returns
429 (rate limit) errors at faster rates.

### Important notes for Claude

- **Never use WebFetch** for Drive files — it has a 15-minute cache
- Use `curl` or `python3 urllib` via Bash tool instead
- The checkpoint JSON can be large (contains all revision entries) — download to `/tmp/` first
- Look at `last_rev` vs `max_rev` for quick progress, or count entries in `revisions` array